# 注意

- 簡便化のため、プロトコルを大胆に改変しています
- 簡便化のため、一部脆弱な実装を含みます
- データフォーマットは独自のものを使っています
- 間違いが含まれてる可能性があります

# 0. 前準備

## 0.1 ルートCA証明書のインポート

In [ ]:
# @title  {"run":"auto"}
ca_crt = "" # @param {"type":"string","placeholder":"CA証明書の中身を貼ってください"}

# crt形式に成形
ca_crt = ca_crt.replace(" ", "\n")
ca_crt = ca_crt.replace("-----BEGIN\nCERTIFICATE-----", "-----BEGIN CERTIFICATE-----")
ca_crt = ca_crt.replace("-----END\nCERTIFICATE-----", "-----END CERTIFICATE-----")
open("ca.crt", "w").write(ca_crt)
print(ca_crt)
print("アップロード完了")

# I. ハンドシェイクプロトコル

## 1. まずは挨拶

### 1.1 クライアントランダムを作ろう！

クライアントハローで送るクライアントランダムを作ろう！

下のセルで
```
!openssl rand -hex 32
```
を実行すると、32バイト(64文字)のランダムな16進数が生成できるよ

In [ ]:
!openssl rand -hex 32

In [ ]:
# @title {"run":"auto"}
client_random = "" # @param {"type":"string","placeholder":"生成できたクライアントランダム"}
print(client_random)

### 1.2 クライアントハローを送ろう！

クライアントハローでは
- 使用可能な暗号スイート
- クライアントランダム

をサーバーに送るよ（簡略化）

暗号スイートは `TLS_RSA_WITH_AES_128_CBC_SHA256` を提案しよう！

例：
```
ClientHelloです
暗号スイート: TLS_RSA_WITH_AES_128_CBC_SHA256
クライアントランダム: （さっき生成した値）
```

### 1.3 サーバーハローを受け取ろう！

サーバーから
- 使用する暗号スイート
- サーバーランダム

が返ってくるよ！受け取ったらサーバーランダムを保存しよう

In [ ]:
# @title {"run":"auto"}
# @markdown サーバーハローを受け取ったらサーバーランダムを保存しよう

server_random = "" # @param {"type":"string","placeholder":"サーバーランダム"}

### 1.4 サーバー証明書を受け取ろう！

サーバーから証明書が送られてくるよ！

受け取ったら保存して、CA証明書で検証しよう

In [ ]:
# @title {"run":"auto"}
# @markdown サーバーの証明書を貼り付けて!
server_crt = "-----BEGIN CERTIFICATE----- … -----END CERTIFICATE-----" # @param {"type":"string","placeholder":"サーバー証明書の中身を貼ってください"}

# crt形式に成形
server_crt = server_crt.replace(" ", "\n")
server_crt = server_crt.replace("-----BEGIN\nCERTIFICATE-----", "-----BEGIN CERTIFICATE-----")
server_crt = server_crt.replace("-----END\nCERTIFICATE-----", "-----END CERTIFICATE-----")
open("server.crt", "w").write(server_crt)
print(server_crt)
print("アップロード完了")

証明書をCA証明書で検証しよう！

```
!openssl verify -CAfile ca.crt server.crt
```
`server.crt: OK` と出ればOK！

フィンガープリントも確認してみよう
```
!openssl x509 -in server.crt -fingerprint -sha256 -noout
```

### 1.5 サーバーハローの終了を確認しよう

サーバーからサーバーハロー終了の合図が来たかな？来たら次に進もう！

## 2. ClientKeyExchange-共通の秘密情報を作る

### 2.1 プレマスターシークレットを作ろう！

プレマスターシークレットは48バイトのランダムな値だよ。

```
!openssl rand 48 > pms.bin
```
で生成できるよ！

16進数で確認するには
```
!od -An -tx1 pms.bin | tr -d ' \n'
```
だ！

### 2.2 プレマスターシークレットを暗号化して送ろう！

プレマスターシークレットはサーバーの公開鍵で暗号化して送るよ。

まずサーバー証明書から公開鍵を取り出そう
```
!openssl x509 -in server.crt -pubkey -noout > server_pub.pem
```

次にプレマスターシークレットを暗号化する
```
!openssl pkeyutl -encrypt \
  -pubin -inkey server_pub.pem \
  -in  pms.bin \
  -out enc_pms.bin
```

base64にしてDiscordで送ろう！
```
!base64 -w 0 enc_pms.bin
```

### 2.3 共通のマスターシークレットを作ろう

プレマスターシークレットを16進数に変換するよ!

`!od -An -tx1 pms.bin | tr -d ' \n'`  
ここではObjectDumpコマンドで`pms.bin`を16進数(TypeHex)にして、その出力から改行を取り除いてるんだ。

In [ ]:
pms = "" # @param {"type":"string","placeholder":"上で出力されたpms"}

In [ ]:
# @markdown 次はサーバーランダムとクライアントランダムを使うよ。

# @markdown 上にさかのぼるのはめんどくさいから、このセルを実行すれば再確認できるようにしといたよ

print(f"client_random     = {client_random}")
print(f"server_random     = {server_random}")

ここでは3つの素材(プレマスターシークレット、サーバーランダム、クライアントランダム)から7つの鍵を作るよ。

- master_secret: 48byte
- client_write_MAC: 32byte
- server_write_MAC: 32byte
- client_write_key: 16byte
- server_write_key: 16byte
- client_write_iv: 16byte
- server_write_iv: 16byte

今回は簡略化のため、シンプルな方法をとるよ。

```python
#hashlib.sha256がstr受け付けないからbyteに変換するよ
base = bytes.fromhex(pms + client_random + server_random)
# sha256の出力が32byteだから2回やる
h1 = hashlib.sha256(base + b"ms1").digest() # bをつけるとbyte列として解釈されるよ
h2 = hashlib.sha256(base + b"ms2").digest()
master_secret = (h1 + h2)[:48].hex() # 最後に16進数文字列に変換
```
これを参考に残りの鍵を作ってみよう!

In [ ]:

import hashlib
#hashlib.sha256がstr受け付けないからbyteに変換するよ
base = bytes.fromhex(pms + client_random + server_random)
# sha256の出力が32byteだから2回やる
h1 = hashlib.sha256(base + b"ms1").digest() # bをつけるとbyte列として解釈されるよ
h2 = hashlib.sha256(base + b"ms2").digest()
master_secret = (h1 + h2)[:48].hex() # 最後に16進数文字列に変換

# 今度は master_secret を base にする
ms_base =  bytes.fromhex(master_secret + client_random + server_random)
# ここに追記
client_write_MAC = hashlib.sha256(ms_base + b"client_write_MAC").digest()[:32].hex()
#server_write_MAC
#client_write_key
#server_write_key
#client_write_iv
#server_write_iv

print(f"master_secret    = {master_secret}")
print(f"client_write_MAC = {client_write_MAC}")
#print(f"server_write_MAC = {server_write_MAC}")
#print(f"client_write_key = {client_write_key}")
#print(f"server_write_key = {server_write_key}")
#print(f"client_write_iv  = {client_write_iv}")
#print(f"server_write_iv  = {server_write_iv}")
print()
print("✓ 鍵導出完了！この値は絶対に Discord に貼らないこと 🔒")

ついでに: Colabの挿入>スクラッチセルを選択し、
```python
print(f"master_secret    = {master_secret}")
print(f"client_write_MAC = {client_write_MAC}")
print(f"server_write_MAC = {server_write_MAC}")
print(f"client_write_key = {client_write_key}")
print(f"server_write_key = {server_write_key}")
print(f"client_write_iv  = {client_write_iv}")
print(f"server_write_iv  = {server_write_iv}")
```
を実行しておくと、この後便利だぞ！

## 3. 暗号文でのやりとりする準備をしよう！

### 3.1 ハンドシェイクのハッシュを求めよう

今までDiscordでやりとりしたハンドシェイクのログをコピペして、ハッシュを求めよう!

ハッシュを求めるのは
```shell
!openssl dgst -sha256 -hex {ハッシュを求めたいファイル} | cut -d' ' -f2
```
だよ！

In [ ]:
# @title ログをファイルに保存する
chat_log = "" # @param {"type":"string","placeholder":"ハンドシェイクチャンネルのログを貼ろう!"}
open("hs_log.txt", "w").write(chat_log)
print("hs_log.txtに保存")

In [ ]:
# @title hs_hash を保存する {"run":"auto"}
hs_hash = "" # @param {"type":"string","placeholder":"openssl dgst の出力（64文字のhex）"}
print(f"hs_hash = {hs_hash}")

### 3.2 クライアントの Finished を送ろう

クライアント側の `hs_hash` を `client_write_key` で暗号化してサーバーに送るよ。

```shell
!echo -n "（さっき求めたhs_hash）" | \
  openssl enc -aes-128-cbc \
  -K "（client_write_key）" -iv "（client_write_iv）" -nosalt | base64 -w 0
```

出力されたbase64をDiscordで送ろう!

例：
```
ChangeCipherSpec です。
Finished（暗号文）:
（base64 の出力をそのまま貼る）

ハンドシェイク以上です。
```

### 3.3 サーバーの Finished を検証しよう

サーバーからも暗号文が返ってくるよ。復号して、自分が求めた `hs_hash` と同じか確認しよう！

```shell
!echo "（サーバーの base64）" | base64 -d | \
  openssl enc -d -aes-128-cbc \
  -K "（server_write_key）" -iv "（server_write_iv）" -nosalt
```

同じハッシュになったかな？なっていれば、お互いが同じ鍵を持っていることが確認できたよ！

# II. 暗号通信開始！アプリケーションデータプロトコル

アプリケーションデータプロトコルのデータは、【平文 + MAC値】を暗号化したものだよ。

MAC値は暗号学入門p.207でやったね。

これを使うと、メッセージが改ざんされたか調べることができるよ。

## 1. 暗号文を送ろう！

必要なのは

- 送りたいメッセージ
- MAC値

の2つだ！

### 1.1 送る文を考えよう！

まずは『送りたいメッセージ』を決めよう！  
明日の予定でも秘密の約束でも、愛の告白でもなんでもいいよ！

### 1.2 MAC値を求めよう！

決めたらMAC値を求めよう

クライアントが書き込むから `client_write_MAC` を使うよ！

```shell
!echo -n "（さっき考えた文）" | openssl dgst -sha256 -mac HMAC \
  -macopt hexkey:"（client_write_MAC）" -hex | cut -d' ' -f2
```

### 1.3 暗号化しよう！

```shell
!echo -e "平文:（さっき考えた文）\nMAC値:（求めたMAC値）" | \
  openssl enc -aes-128-cbc \
  -K "（client_write_key）" -iv "（client_write_iv）" -nosalt | base64 -w 0
```

出力されたbase64をDiscordで送ろう！

## 2. 暗号文を受け取ろう！

サーバーからも暗号文が送られてくるよ。復号して検証してみよう！

### 2.1 暗号文を復号する

```shell
!echo "（サーバーの base64）" | base64 -d | \
  openssl enc -d -aes-128-cbc \
  -K "（server_write_key）" -iv "（server_write_iv）" -nosalt
```

```
平文:
MAC値:
```
の形式になったかな？

### 2.2 メッセージの検証

MAC値が正しいか検証しよう！

今度はサーバーが書き込んだから `server_write_MAC` を使うよ！

```shell
!echo -n "（平文）" | openssl dgst -sha256 -mac HMAC \
  -macopt hexkey:"（server_write_MAC）" -hex | cut -d' ' -f2
```

同じMAC値になったかな？

# III. 完了！

おつかれさま！！

パケットはみんなに見えていたのに、内容はバレなかったよね。

これがTLSだよ！